In [0]:
# Databricks Notebook
# 02_silver_transform

from pyspark.sql.functions import (
    col,
    round,
    from_json,
    explode,
    when,
    regexp_replace,
    trim
)
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

# Read Bronze table
bronze_df = spark.table("smart_carparking.bronze_carpark_raw")

# Define JSON schema
carpark_schema = StructType([
    StructField("tsn", StringType(), True),
    StructField("time", StringType(), True),
    StructField("spots", StringType(), True),
    StructField("ParkID", StringType(), True),
    StructField("MessageDate", StringType(), True),
    StructField("facility_id", StringType(), True),
    StructField("facility_name", StringType(), True),
    StructField("tfnsw_facility_id", StringType(), True),

    StructField("location", StructType([
        StructField("suburb", StringType(), True),
        StructField("address", StringType(), True),
        StructField("latitude", StringType(), True),
        StructField("longitude", StringType(), True),
    ]), True),

    StructField("occupancy", StructType([
        StructField("loop", StringType(), True),
        StructField("total", StringType(), True),
        StructField("monthlies", StringType(), True),
        StructField("open_gate", StringType(), True),
        StructField("transients", StringType(), True),
    ]), True),

    StructField("zones", ArrayType(StructType([
        StructField("spots", StringType(), True),
        StructField("zone_id", StringType(), True),
        StructField("zone_name", StringType(), True),
        StructField("parent_zone_id", StringType(), True),
        StructField("occupancy", StructType([
            StructField("loop", StringType(), True),
            StructField("total", StringType(), True),
            StructField("monthlies", StringType(), True),
            StructField("open_gate", StringType(), True),
            StructField("transients", StringType(), True),
        ]), True),
    ])), True)
])

# Parse Bronze raw_json
parsed_df = bronze_df.withColumn(
    "data",
    from_json(col("raw_json"), carpark_schema)
)

# Clean facility name expression
clean_facility_name = trim(
    regexp_replace(
        col("data").getField("facility_name"),
        "Park&Ride - ",
        ""
    )
)

# ======================================================================================
# Part 1 - Facility-level Silver table
# One row per car park facility per API snapshot
# ======================================================================================

silver_facility_df = parsed_df.select(
    col("data").getField("facility_id").alias("facility_id"),
    clean_facility_name.alias("facility_name"),
    col("data").getField("tfnsw_facility_id").alias("tfnsw_facility_id"),
    col("data").getField("tsn").alias("tsn"),
    col("data").getField("time").alias("api_time_seconds_since_2000"),
    col("data").getField("ParkID").cast("int").alias("park_id"),

    col("data").getField("location").getField("suburb").alias("suburb"),
    col("data").getField("location").getField("address").alias("address"),
    col("data").getField("location").getField("latitude").cast("double").alias("latitude"),
    col("data").getField("location").getField("longitude").cast("double").alias("longitude"),

    col("data").getField("spots").cast("int").alias("total_spots"),
    col("data").getField("occupancy").getField("total").cast("int").alias("occupied_spaces"),
    col("data").getField("occupancy").getField("loop").cast("int").alias("loop_count"),
    col("data").getField("occupancy").getField("transients").cast("int").alias("transient_vehicles"),
    col("data").getField("occupancy").getField("monthlies").cast("int").alias("monthly_vehicles"),
    col("data").getField("occupancy").getField("open_gate").cast("int").alias("open_gate_count"),

    col("data").getField("MessageDate").cast("timestamp").alias("message_datetime"),
    col("ingestion_timestamp"),
    col("source_system")
)

silver_facility_df = (
    silver_facility_df
    .withColumn(
        "available_spaces",
        when(
            col("occupied_spaces").isNotNull(),
            col("total_spots") - col("occupied_spaces")
        ).otherwise(None)
    )
    .withColumn(
        "occupancy_rate",
        when(
            (col("total_spots") > 0) & (col("occupied_spaces").isNotNull()),
            round((col("occupied_spaces") / col("total_spots")) * 100, 2)
        ).otherwise(None)
    )
)

silver_facility_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("smart_carparking.silver_carpark_occupancy")

display(silver_facility_df)

# ======================================================================================
# Part 2 - Zone-level Silver table
# One row per internal zone per car park facility per API snapshot
# ======================================================================================

silver_zones_df = (
    parsed_df
    .withColumn("zone", explode(col("data").getField("zones")))
    .select(
        col("data").getField("facility_id").alias("facility_id"),
        clean_facility_name.alias("facility_name"),
        col("data").getField("tfnsw_facility_id").alias("tfnsw_facility_id"),
        col("data").getField("tsn").alias("tsn"),
        col("data").getField("time").alias("api_time_seconds_since_2000"),
        col("data").getField("ParkID").cast("int").alias("park_id"),

        col("data").getField("location").getField("suburb").alias("suburb"),
        col("data").getField("location").getField("address").alias("address"),
        col("data").getField("MessageDate").cast("timestamp").alias("message_datetime"),

        col("zone").getField("zone_id").alias("zone_id"),
        col("zone").getField("zone_name").alias("zone_name"),
        col("zone").getField("parent_zone_id").alias("parent_zone_id"),
        col("zone").getField("spots").cast("int").alias("zone_total_spots"),

        col("zone").getField("occupancy").getField("total").cast("int").alias("zone_occupied_spaces"),
        col("zone").getField("occupancy").getField("loop").cast("int").alias("zone_loop_count"),
        col("zone").getField("occupancy").getField("transients").cast("int").alias("zone_transient_vehicles"),
        col("zone").getField("occupancy").getField("monthlies").cast("int").alias("zone_monthly_vehicles"),
        col("zone").getField("occupancy").getField("open_gate").cast("int").alias("zone_open_gate_count"),

        col("ingestion_timestamp"),
        col("source_system")
    )
)

silver_zones_df = (
    silver_zones_df
    .withColumn(
        "zone_available_spaces",
        when(
            col("zone_occupied_spaces").isNotNull(),
            col("zone_total_spots") - col("zone_occupied_spaces")
        ).otherwise(None)
    )
    .withColumn(
        "zone_occupancy_rate",
        when(
            (col("zone_total_spots") > 0) & (col("zone_occupied_spaces").isNotNull()),
            round((col("zone_occupied_spaces") / col("zone_total_spots")) * 100, 2)
        ).otherwise(None)
    )
    .withColumn(
        "zone_loop_rate",
        when(
            (col("zone_total_spots") > 0) & (col("zone_loop_count").isNotNull()),
            round((col("zone_loop_count") / col("zone_total_spots")) * 100, 2)
        ).otherwise(None)
    )
)

silver_zones_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("smart_carparking.silver_carpark_zones")

display(silver_zones_df)

facility_id,facility_name,tfnsw_facility_id,tsn,api_time_seconds_since_2000,park_id,suburb,address,latitude,longitude,total_spots,occupied_spaces,loop_count,transient_vehicles,monthly_vehicles,open_gate_count,message_datetime,ingestion_timestamp,source_system,available_spaces,occupancy_rate
486,Ashfield,213110TPR001,213110,831502008,1,Ashfield,Brown Street,-33.888104,151.126577,216,42,null,50,0,0,2026-05-07T20:46:48.000Z,2026-05-07T10:52:11.171Z,TfNSW Car Park API,174,19.44
487,Kogarah,221710TPR001,221710,831501965,3,Kogarah,2 Railway Street,-33.96369941,151.1319494,259,27,null,20,0,0,2026-05-07T20:46:05.000Z,2026-05-07T10:52:11.171Z,TfNSW Car Park API,232,10.42
488,Seven Hills,214710TPR001,214710,831502231,4,Seven Hills,Terminus Road,-33.77304548,150.9367514,1613,0,null,0,0,0,2026-05-07T20:50:31.000Z,2026-05-07T10:52:11.171Z,TfNSW Car Park API,1613,0.0
489,Manly Vale,2093117TPR001,2093117,831502082,1,Manly Vale,84 Kenneth Road,-33.786536,151.267221,142,33,null,0,0,0,2026-05-07T20:48:02.000Z,2026-05-07T10:52:11.171Z,TfNSW Car Park API,109,23.24
6,Gordon Henry St (north),207210TPR001,207210,831466150,1,Gordon,Henry Street,-33.757065,151.154662,213,23,null,null,null,null,2026-05-07T20:49:10.000Z,2026-05-07T10:52:11.171Z,TfNSW Car Park API,190,10.8
7,Kiama,253330TPR001,253330,831466325,1,Kiama,Bong Bong Street,-34.673122,150.854546,42,1,null,null,null,null,2026-05-07T20:52:05.000Z,2026-05-07T10:52:11.171Z,TfNSW Car Park API,41,2.38
9,Revesby,221210TPR001,221210,831466253,1,Revesby,The River Road,-33.95107517,151.0168491,931,81,595822,null,null,null,2026-05-07T20:50:53.000Z,2026-05-07T10:52:11.171Z,TfNSW Car Park API,850,8.7
11,Narrabeen,2101130TPR001,2101130,831465897,1,Narrabeen,Pittwater Road,-33.714364,151.29699,46,3,153853,null,null,null,2026-05-07T20:44:57.000Z,2026-05-07T10:52:11.171Z,TfNSW Car Park API,43,6.52
13,Dee Why,2099207TPR001,2099207,831466101,1,Dee Why,40 Kingsway,-33.750302,151.286717,121,8,164699,null,null,null,2026-05-07T20:48:21.000Z,2026-05-07T10:52:11.171Z,TfNSW Car Park API,113,6.61
12,Mona Vale,2103108TPR001,2103108,831463870,1,Mona Vale,Golf Avenue,-33.677567,151.306512,68,6,null,null,null,null,2026-05-07T20:11:10.000Z,2026-05-07T10:52:11.171Z,TfNSW Car Park API,62,8.82


facility_id,facility_name,tfnsw_facility_id,tsn,api_time_seconds_since_2000,park_id,suburb,address,message_datetime,zone_id,zone_name,parent_zone_id,zone_total_spots,zone_occupied_spaces,zone_loop_count,zone_transient_vehicles,zone_monthly_vehicles,zone_open_gate_count,ingestion_timestamp,source_system,zone_available_spaces,zone_occupancy_rate,zone_loop_rate
486,Ashfield,213110TPR001,213110,831502008,1,Ashfield,Brown Street,2026-05-07T20:46:48.000Z,1,MainArea,0,216,null,42,47,0,null,2026-05-07T10:52:11.171Z,TfNSW Car Park API,null,null,19.44
487,Kogarah,221710TPR001,221710,831501965,3,Kogarah,2 Railway Street,2026-05-07T20:46:05.000Z,2,MainArea,0,259,null,27,14,0,null,2026-05-07T10:52:11.171Z,TfNSW Car Park API,null,null,10.42
488,Seven Hills,214710TPR001,214710,831502231,4,Seven Hills,Terminus Road,2026-05-07T20:50:31.000Z,4,Multi Level,0,874,null,696,253,0,null,2026-05-07T10:52:11.171Z,TfNSW Car Park API,null,null,79.63
488,Seven Hills,214710TPR001,214710,831502231,4,Seven Hills,Terminus Road,2026-05-07T20:50:31.000Z,5,on grade,0,739,null,191,140,0,null,2026-05-07T10:52:11.171Z,TfNSW Car Park API,null,null,25.85
488,Seven Hills,214710TPR001,214710,831502231,4,Seven Hills,Terminus Road,2026-05-07T20:50:31.000Z,0,,0,0,null,0,0,0,null,2026-05-07T10:52:11.171Z,TfNSW Car Park API,null,null,null
488,Seven Hills,214710TPR001,214710,831502231,4,Seven Hills,Terminus Road,2026-05-07T20:50:31.000Z,0,,0,0,null,0,0,0,null,2026-05-07T10:52:11.171Z,TfNSW Car Park API,null,null,null
488,Seven Hills,214710TPR001,214710,831502231,4,Seven Hills,Terminus Road,2026-05-07T20:50:31.000Z,0,,0,0,null,0,0,0,null,2026-05-07T10:52:11.171Z,TfNSW Car Park API,null,null,null
488,Seven Hills,214710TPR001,214710,831502231,4,Seven Hills,Terminus Road,2026-05-07T20:50:31.000Z,0,,0,0,null,0,0,0,null,2026-05-07T10:52:11.171Z,TfNSW Car Park API,null,null,null
489,Manly Vale,2093117TPR001,2093117,831502082,1,Manly Vale,84 Kenneth Road,2026-05-07T20:48:02.000Z,1,MainArea,0,142,null,33,0,0,null,2026-05-07T10:52:11.171Z,TfNSW Car Park API,null,null,23.24
6,Gordon Henry St (north),207210TPR001,207210,831466150,1,Gordon,Henry Street,2026-05-07T20:49:10.000Z,1,Park&Ride - Gordon Henry St (north),0,213,23,null,null,null,null,2026-05-07T10:52:11.171Z,TfNSW Car Park API,190,10.8,null
